In [1]:
import ee
import geemap
import os
from google.colab import files
import geopandas as gpd
import json
import numpy as np
from glob import glob
from google.colab import drive
import matplotlib.pyplot as plt

In [2]:
# Authenticate and initialize
ee.Authenticate()
ee.Initialize(project='ee-rubelahmmed01')

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_7TDKVSyKvBdmMqW?ref=4i2o6


In [3]:
# Define the desired directory path
target_directory = '/content/Shape_File'

uploaded = files.upload(target_directory)

Saving AOI_Meghna_Chandpur_DS.geojson to /content/Shape_File/AOI_Meghna_Chandpur_DS.geojson


In [4]:
# Load GeoJSON
with open("/content/Shape_File/AOI_Meghna_Chandpur_DS.geojson") as f:
    geojson_data = json.load(f)

# Convert to EE Geometry or FeatureCollection
roi = ee.FeatureCollection(geojson_data)

# **✅ Landsat 5 (LT05) — Post-Monsoon Water Masks**

In [5]:
def scale_landsat5_red(img):
    nir = img.select('SR_B4')
    scaled_red = nir.multiply(0.0000275).add(-0.2).rename('NIR')
    return img.addBands(scaled_red, overwrite=True)


def get_ls5_red_band(year):
    start = f"{year-1}-12-01"
    end = f"{year}-03-01"

    col = (ee.ImageCollection("LANDSAT/LT05/C02/T1_L2")
           .filterBounds(roi)
           .filterDate(start, end)
           .filter(ee.Filter.lt("CLOUD_COVER", 20))
           .map(scale_landsat5_red))

    if col.size().getInfo() == 0:
        print(f"No images for LS5 year {year}, skipping...")
        return None

    median = col.median().clip(roi)
    red_band = median.select('NIR')
    water_mask = red_band.updateMask(red_band.lt(0.15)).rename('water_mask')
    return water_mask.set({'year': year, 'source': 'LS5'})

# Collect red bands
ls5_NIR_bands = []
for year in range(1986, 2014):
    red = get_ls5_red_band(year)
    if red:
        ls5_NIR_bands.append(red)


No images for LS5 year 1986, skipping...
No images for LS5 year 1987, skipping...
No images for LS5 year 2002, skipping...
No images for LS5 year 2003, skipping...
No images for LS5 year 2008, skipping...
No images for LS5 year 2012, skipping...
No images for LS5 year 2013, skipping...


In [8]:
import matplotlib.pyplot as plt

# Generate 24 distinct colors using matplotlib's tab20 and tab10 colormaps
colors = plt.cm.tab20.colors + plt.cm.tab10.colors  # total 30 colors

# Convert RGB tuples to hex format
palette = ['#{:02x}{:02x}{:02x}'.format(int(r*255), int(g*255), int(b*255)) for r, g, b in colors[:30]]

print(palette)

# Visualize LS5 masks on a map
Map = geemap.Map()
Map.centerObject(roi, 10) # Center the map on the ROI
Map.add_layer(roi, {}, 'AOI')

# Add each LS5 water mask to the map
for i, img in enumerate(ls5_NIR_bands):
    year = img.get('year').getInfo()
    color = palette[i % len(palette)]
    Map.addLayer(img, {'palette': [color]}, f'Water Mask {year}')

Map

['#1f77b4', '#aec7e8', '#ff7f0e', '#ffbb78', '#2ca02c', '#98df8a', '#d62728', '#ff9896', '#9467bd', '#c5b0d5', '#8c564b', '#c49c94', '#e377c2', '#f7b6d2', '#7f7f7f', '#c7c7c7', '#bcbd22', '#dbdb8d', '#17becf', '#9edae5', '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']


Map(center=[23.048291158892276, 90.58817965268534], controls=(WidgetControl(options=['position', 'transparent_…

In [7]:
# Create a geemap Map
Map = geemap.Map()
Map.centerObject(roi, 10)  # Center map on ROI
Map.add_layer(roi, {}, 'AOI')

# Visualization parameters for Red band (grayscale)
vis_params = {
    'min': 0.0,
    'max': 0.30,  # Adjust as needed for scaled Red reflectance
}

# Add each year's Red band image to the map
for img in ls5_NIR_bands:
    year = img.get('year').getInfo()
    Map.addLayer(img, vis_params, f'NIR Band {year}')

Map


Map(center=[23.048291158892276, 90.58817965268534], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
import time
for img in ls5_NIR_bands:
    year = img.get("year").getInfo()
    task = ee.batch.Export.image.toDrive(
        image=img,
        description=f"{year}_LS5_NIR_Band",
        folder="LS5_NIR_Band_1988_2012",
        fileNamePrefix=f"{year}_LS5_NIR_Band_1988_2012",
        region=roi.geometry().bounds().getInfo()["coordinates"],
        scale=30,
        crs='EPSG:32645',
        maxPixels=1e13
    )
    task.start()
    print(f"🚀 LS5 export started for {year}")
    time.sleep(20)

🚀 LS5 export started for 1988
🚀 LS5 export started for 1989
🚀 LS5 export started for 1990
🚀 LS5 export started for 1991
🚀 LS5 export started for 1992
🚀 LS5 export started for 1993
🚀 LS5 export started for 1994
🚀 LS5 export started for 1995
🚀 LS5 export started for 1996
🚀 LS5 export started for 1997
🚀 LS5 export started for 1998
🚀 LS5 export started for 1999
🚀 LS5 export started for 2000
🚀 LS5 export started for 2001
🚀 LS5 export started for 2004
🚀 LS5 export started for 2005
🚀 LS5 export started for 2006
🚀 LS5 export started for 2007
🚀 LS5 export started for 2009
🚀 LS5 export started for 2010
🚀 LS5 export started for 2011


# **✅ Water Mask (NDWI) Export Landsat 7**

In [ ]:
def scale_landsat7_nir(img):
    nir = img.select('SR_B4')
    scaled_nir = nir.multiply(0.0000275).add(-0.2).rename('NIR')
    return img.addBands(scaled_nir, overwrite=True)

def get_ls7_nir_band(year):
    start = f"{year-1}-12-01"
    end = f"{year}-03-01"

    col = (ee.ImageCollection("LANDSAT/LE07/C02/T1_L2")
           .filterBounds(roi)
           .filterDate(start, end)
           .filter(ee.Filter.lt("CLOUD_COVER", 1))
           .map(scale_landsat7_nir))

    if col.size().getInfo() == 0:
        print(f"No images for LS7 year {year}, skipping...")
        return None

    median = col.median().clip(roi)
    nir_band = median.select('NIR')
    return nir_band.set({'year': year, 'source': 'LS7'})

# Collect LS7 NIR bands
ls7_NIR_bands = []
for year in range(1999, 2014):  # Landsat 7 launched in 1999
    nir = get_ls7_nir_band(year)
    if nir:
        ls7_NIR_bands.append(nir)


No images for LS7 year 1999, skipping...


In [ ]:
# Create a geemap Map
Map = geemap.Map()
Map.centerObject(roi, 10)  # Center map on ROI
Map.add_layer(roi, {}, 'AOI')

# Visualization parameters for Red band (grayscale)
vis_params = {
    'min': 0.0,
    'max': 0.30,  # Adjust as needed for scaled Red reflectance
}

# Add each year's Red band image to the map
for img in ls7_NIR_bands:
    year = img.get('year').getInfo()
    Map.addLayer(img, vis_params, f'NIR Band {year}')

Map

Map(center=[23.048291158892276, 90.58817965268534], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
import time
for img in ls7_NIR_bands:
    year = img.get("year").getInfo()
    task = ee.batch.Export.image.toDrive(
        image=img,
        description=f"{year}_LS7_NIR_Band",
        folder="LS7_NIR_Band_2000_2024",
        fileNamePrefix=f"{year}_LS7_NIR_Band",
        region=roi.geometry().bounds().getInfo()["coordinates"],
        scale=30,
        crs='EPSG:32645',
        maxPixels=1e13
    )
    task.start()
    print(f"🚀 LS5 export started for {year}")
    time.sleep(20)


🚀 LS5 export started for 2000
🚀 LS5 export started for 2001
🚀 LS5 export started for 2002
🚀 LS5 export started for 2003
🚀 LS5 export started for 2004
🚀 LS5 export started for 2005
🚀 LS5 export started for 2006
🚀 LS5 export started for 2007
🚀 LS5 export started for 2008
🚀 LS5 export started for 2009
🚀 LS5 export started for 2010
🚀 LS5 export started for 2011
🚀 LS5 export started for 2012
🚀 LS5 export started for 2013


# ✅ ** Landsat 8 (LC08) — Post-Monsoon Water Masks (2014–2024)**

In [ ]:
def scale_landsat8_nir(img):
    nir = img.select('SR_B5')  # NIR for Landsat 8
    scaled_nir = nir.multiply(0.0000275).add(-0.2).rename('NIR')
    return img.addBands(scaled_nir, overwrite=True)

def get_ls8_nir_band(year):
    start = f"{year-1}-12-01"
    end = f"{year}-03-01"

    col = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
           .filterBounds(roi)
           .filterDate(start, end)
           .filter(ee.Filter.lt("CLOUD_COVER", 20))
           .map(scale_landsat8_nir))

    if col.size().getInfo() == 0:
        print(f"No images for LS8 year {year}, skipping...")
        return None

    median = col.median().clip(roi)
    nir_band = median.select('NIR')
    return nir_band.set({'year': year, 'source': 'LS8'})

# Collect LS8 NIR bands
ls8_NIR_bands = []
for year in range(2014, 2025):  # 2014 to 2025 inclusive
    nir = get_ls8_nir_band(year)
    if nir:
        ls8_NIR_bands.append(nir)


In [ ]:
# Create a geemap Map
Map = geemap.Map()
Map.centerObject(roi, 10)  # Center map on ROI
Map.add_layer(roi, {}, 'AOI')

# Visualization parameters for Red band (grayscale)
vis_params = {
    'min': 0.0,
    'max': 0.30,  # Adjust as needed for scaled Red reflectance
}

# Add each year's Red band image to the map
for img in ls8_NIR_bands:
    year = img.get('year').getInfo()
    Map.addLayer(img, vis_params, f'NIR Band {year}')

Map

Map(center=[23.048291158892276, 90.58817965268534], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
import time
for img in ls8_NIR_bands:
    year = img.get("year").getInfo()
    task = ee.batch.Export.image.toDrive(
        image=img,
        description=f"{year}_LS8_NIR_Band",
        folder="LS8_NIR_Band_2000_2024",
        fileNamePrefix=f"{year}_LS8_NIR_Band",
        region=roi.geometry().bounds().getInfo()["coordinates"],
        scale=30,
        crs='EPSG:32645',
        maxPixels=1e13
    )
    task.start()
    print(f"🚀 LS5 export started for {year}")
    time.sleep(20)


🚀 LS5 export started for 2014
🚀 LS5 export started for 2015
🚀 LS5 export started for 2016
🚀 LS5 export started for 2017
🚀 LS5 export started for 2018
🚀 LS5 export started for 2019
🚀 LS5 export started for 2020
🚀 LS5 export started for 2021
🚀 LS5 export started for 2022
🚀 LS5 export started for 2023
🚀 LS5 export started for 2024
